In [ ]:
from pathlib import Path
import pandas as pd

def find_data_dir():
    for parent in [Path.cwd(), *Path.cwd().parents]:
        for candidate in [parent / "data", parent / "fire_risk_project" / "data"]:
            if (candidate / "08_점수규칙DB.csv").is_file():
                return candidate
    raise FileNotFoundError("프로젝트 data 폴더를 찾을 수 없습니다.")

data_dir = find_data_dir()

# 현재 data 폴더의 CSV를 직접 읽습니다.
tasks = pd.read_csv(data_dir / "01_공정_세부작업DB.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
questions = pd.read_csv(data_dir / "02_위험요인_입력항목DB.csv", encoding="utf-8-sig", dtype=str, keep_default_na=False)
print("읽은 폴더:", data_dir)
print("작업 수:", len(tasks))
print("질문 수:", len(questions))
display(questions.head())


In [ ]:
# 기존 CSV를 사용할 때는 False로 유지합니다.
# 엑셀을 다시 변환하려면 아래 경로에 원본을 넣고 True로 바꾸세요.
CONVERT_EXCEL_TO_CSV = False
excel_path = data_dir / "건설현장_화재폭발_DB_v1.1_구조정비본.xlsx"

if CONVERT_EXCEL_TO_CSV:
    if not excel_path.is_file():
        raise FileNotFoundError(f"엑셀 파일이 없습니다: {excel_path}")
    from datetime import datetime
    import shutil

    # 현재 data 폴더에 CSV 저장

    sheet_names = [
        "01_공정_세부작업DB",
        "02_위험요인_입력항목DB",
        "03_사고데이터",
        "04_법령_KOSHA기준DB",
        "05_MSDS_물질DB",
        "06_현장입력_결과",
        "06_현장응답상세",
        "07_기상데이터",
        "08_점수규칙DB",
        "99_코드표",
    ]

    # 모든 시트를 먼저 읽기
    tables = pd.read_excel(excel_path, sheet_name=sheet_names)

    # 실제 기록이 있는 행만 남길 때 사용할 ID
    record_ids = {
        "06_현장입력_결과": "평가ID",
        "06_현장응답상세": "평가ID",
        "07_기상데이터": "기상ID",
    }

    for sheet_name, df in tables.items():
        df = df.dropna(how="all")

        if sheet_name in record_ids:
            id_column = record_ids[sheet_name]
            has_id = (
                df[id_column].notna()
                & df[id_column].astype(str).str.strip().ne("")
            )
            df = df.loc[has_id]

        tables[sheet_name] = df

    # 덮어쓸 기존 CSV만 백업
    backup_dir = (
        data_dir
        / "backups"
        / datetime.now().strftime("%Y%m%d_%H%M%S_%f")
    )
    backup_dir.mkdir(parents=True, exist_ok=False)

    for sheet_name in sheet_names:
        csv_path = data_dir / f"{sheet_name}.csv"

        if csv_path.exists():
            shutil.copy2(csv_path, backup_dir / csv_path.name)

    print("백업 위치:", backup_dir)

    # Excel에서도 한글이 잘 열리도록 UTF-8 BOM으로 저장
    for sheet_name, df in tables.items():
        csv_path = data_dir / f"{sheet_name}.csv"
        df.to_csv(csv_path, index=False, encoding="utf-8-sig")

        print(f"{csv_path.name}: {len(df)}행 저장")
else:
    print("엑셀 변환을 건너뛰고 기존 CSV를 사용합니다:", data_dir)


In [3]:
# CSV에서 ID와 선택지 등을 문자열로 읽기
# 빈칸은 "nan" 대신 빈 문자열로 유지
tasks = pd.read_csv(
    data_dir / "01_공정_세부작업DB.csv",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
)

questions = pd.read_csv(
    data_dir / "02_위험요인_입력항목DB.csv",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
)


def get_questions_for_task(task_id):
    """작업ID에 연결된 질문 목록을 반환합니다."""

    task_id = task_id.strip()

    # 존재하는 작업인지 확인
    if task_id not in tasks["작업ID"].str.strip().values:
        raise ValueError(f"존재하지 않는 작업ID입니다: {task_id}")

    # 예: "H05;H06;H08"을 개별 작업ID 목록으로 분리
    applies_to_task = questions["적용작업ID"].apply(
        lambda value: task_id in [
            item.strip() for item in value.split(";")
        ]
    )

    output_columns = [
        "항목ID",
        "확인항목",
        "입력형식",
        "선택지·허용값",
        "단위",
        "필수여부",
        "점수규칙ID",
        "연결기준ID",
    ]

    return (
        questions.loc[applies_to_task, output_columns]
        .copy()
        .reset_index(drop=True)
    )


# H05 작업의 질문 확인
h05_questions = get_questions_for_task("H05")

print("H05 질문 수:", len(h05_questions))
display(h05_questions)

H05 질문 수: 9


,항목ID,확인항목,입력형식,선택지·허용값,단위,필수여부,점수규칙ID,연결기준ID
0,Q001,주변 가연물 존재 여부,선택,있음 / 없음,-,1,S001,R001
1,Q002,가연물과 작업점 거리,숫자,0 이상,m,1,S002,R001;R002
2,Q003,불티 비산방지조치(용접방화포 등) 실시 여부,선택,실시 / 미실시 / 해당없음,-,1,S003,R001
3,Q004,화재감시자 배치 여부,선택,배치 / 미배치 / 해당없음,-,1,S004,R002
4,Q018,용접방화포 사용 시 성능인증 제품 여부,선택,인증제품 / 미인증제품 / 용접방화포 미사용,-,1,S019,R001
5,Q019,작업점 하부 가연물의 불티 착화 우려 여부,선택,있음 / 없음 / 확인불가,-,1,S020,R002
6,Q020,금속 칸막이·벽·천장·지붕 반대면 가연물의 열전도·복사 착화 우려 여부,선택,있음 / 없음 / 확인불가,-,1,S021,R002
7,Q025,초기소화설비 비치·접근 가능 여부,선택,적정 / 부적정 / 미비치 / 확인불가,-,1,S026,R001;R014
8,Q026,작업 종료 후 잔불·불티 확인 여부,선택,완료 / 미완료 / 확인불가,-,1,S027,R015


In [4]:
import math

# 점수규칙 CSV 읽기
score_rules = pd.read_csv(
    data_dir / "08_점수규칙DB.csv",
    encoding="utf-8-sig",
    dtype=str,
    keep_default_na=False,
)


def score_answer(question_id, answer):
    """선택형 질문의 응답을 점수와 가중치로 변환합니다."""

    question_id = str(question_id).strip()
    answer = str(answer).strip()

    # 1. 질문 찾기
    matched_questions = questions.loc[
        questions["항목ID"].str.strip() == question_id
    ]

    if len(matched_questions) != 1:
        raise ValueError(
            f"질문ID가 없거나 중복되어 있습니다: {question_id}"
        )

    question = matched_questions.iloc[0]

    if question["입력형식"].strip() != "선택":
        raise NotImplementedError(
            "현재 함수는 선택형 질문만 처리합니다."
        )

    # 2. 허용된 응답인지 확인
    allowed_answers = [
        value.strip()
        for value in question["선택지·허용값"].split("/")
    ]

    if answer not in allowed_answers:
        raise ValueError(
            f"{question_id}의 허용 응답: {allowed_answers}"
        )

    # 3. 질문에 연결된 규칙에서 응답과 일치하는 행 찾기
    rule_id = question["점수규칙ID"].strip()

    matched_rules = score_rules.loc[
        (score_rules["점수규칙ID"].str.strip() == rule_id)
        & (score_rules["규칙유형"].str.strip() == "범주형")
        & (score_rules["비교연산자"].str.strip() == "EQ")
        & (score_rules["입력값·조건"].str.strip() == answer)
    ]

    if len(matched_rules) != 1:
        raise ValueError(
            f"일치하는 규칙이 없거나 중복됩니다: {rule_id}, {answer}"
        )

    rule = matched_rules.iloc[0]

    # 4. CSV의 점수·가중치·정규화 기준 읽기
    score = float(rule["점수"])
    weight = float(rule["가중치"])
    max_score = float(rule["정규화최대점수"])

    if not all(math.isfinite(x) for x in [score, weight, max_score]):
        raise ValueError("점수·가중치·정규화 기준에 잘못된 값이 있습니다.")

    if max_score <= 0 or weight < 0 or not 0 <= score <= max_score:
        raise ValueError("점수 또는 가중치가 허용 범위를 벗어났습니다.")

    # 가중치 0인 분기용·해당없음 응답은 합산에서 제외
    included = weight > 0

    return {
        "항목ID": question_id,
        "확인항목": question["확인항목"],
        "사용자응답": answer,
        "점수규칙ID": rule_id,
        "조건순번": rule["조건순번"],
        "점수": score,
        "가중치": weight,
        "가중점수": score * weight if included else 0,
        "최대가중점수": max_score * weight if included else 0,
        "점수합산대상": included,
        "적용조건": rule["적용조건"],
        "확정상태": rule["확정상태"],
    }

In [5]:
def calculate_risk_index(answers):
    """
    answers 예시:
    {"Q003": "미실시", "Q004": "미배치"}

    현재는 선택형 응답의 합산 계산을 검증하는 함수입니다.
    작업별 필수응답·적용조건·중복집계 검사는 아직 포함하지 않습니다.
    """

    if not answers:
        raise ValueError("응답을 한 개 이상 입력하세요.")

    # 각 응답을 점수로 변환
    results = []

    for question_id, answer in answers.items():
        result = score_answer(question_id, answer)
        results.append(result)

    detail = pd.DataFrame(results)

    # 계산에 포함되는 응답만 합산
    included = detail.loc[detail["점수합산대상"]]

    total_score = included["가중점수"].sum()
    max_score = included["최대가중점수"].sum()

    # 분모가 0이면 위험도를 계산할 수 없음
    if max_score == 0:
        risk_index = None
        status = "계산 불가: 합산 대상 없음"
    else:
        risk_index = total_score / max_score * 100
        status = "부분응답 계산 — 최종 현장평가 아님"

    summary = {
        "총가중위험점수": total_score,
        "최대가중점수": max_score,
        "위험지수": risk_index,
        "계산상태": status,
    }

    return summary, detail

In [6]:
# 불티 비산방지조치는 미실시, 화재감시자는 배치한 예시
answers = {
    "Q003": "미실시",
    "Q004": "배치",
}

summary, detail = calculate_risk_index(answers)

display(detail)
display(pd.DataFrame([summary]))

,항목ID,확인항목,사용자응답,점수규칙ID,조건순번,점수,가중치,가중점수,최대가중점수,점수합산대상,적용조건,확정상태
0,Q003,불티 비산방지조치(용접방화포 등) 실시 여부,미실시,S003,1,3.0,1.5,4.5,4.5,True,화기·불티 작업,미확정
1,Q004,화재감시자 배치 여부,배치,S004,2,0.0,2.0,0.0,6.0,True,법령상 배치대상 조건 충족 시 적용,미확정


,총가중위험점수,최대가중점수,위험지수,계산상태
0,4.5,10.5,42.857143,부분응답 계산 — 최종 현장평가 아님


In [7]:
def check_required_answers(task_id, answers):
    """선택한 작업의 필수응답 누락과 잘못 들어온 질문을 확인합니다."""

    # 해당 작업의 질문 목록
    task_questions = get_questions_for_task(task_id)

    # CSV의 필수여부를 숫자로 변환
    required_flags = pd.to_numeric(
        task_questions["필수여부"],
        errors="raise",
    )

    if not required_flags.isin([0, 1]).all():
        raise ValueError("필수여부에 0 또는 1이 아닌 값이 있습니다.")

    required_questions = task_questions.loc[
        required_flags == 1
    ]

    # 입력되지 않았거나 빈칸인 필수응답 찾기
    missing_rows = []

    for _, question in required_questions.iterrows():
        question_id = question["항목ID"]
        answer = answers.get(question_id)

        if pd.isna(answer) or str(answer).strip() == "":
            missing_rows.append({
                "항목ID": question_id,
                "확인항목": question["확인항목"],
            })

    missing = pd.DataFrame(
        missing_rows,
        columns=["항목ID", "확인항목"],
    )

    # 다른 작업의 질문이 섞였는지 확인
    valid_ids = set(task_questions["항목ID"])
    unexpected_ids = sorted(set(answers) - valid_ids)

    summary = {
        "작업ID": task_id,
        "필수질문수": len(required_questions),
        "누락질문수": len(missing),
        "작업외질문ID": unexpected_ids,
        "필수응답입력완료": (
            missing.empty and not unexpected_ids
        ),
    }

    return summary, missing

In [8]:
answers = {
    "Q003": "미실시",
    "Q004": "배치",
}

check_result, missing_questions = check_required_answers(
    "H05",
    answers,
)

display(pd.DataFrame([check_result]))
display(missing_questions)

,작업ID,필수질문수,누락질문수,작업외질문ID,필수응답입력완료
0,H05,9,7,[],False


,항목ID,확인항목
0,Q001,주변 가연물 존재 여부
1,Q002,가연물과 작업점 거리
2,Q018,용접방화포 사용 시 성능인증 제품 여부
3,Q019,작업점 하부 가연물의 불티 착화 우려 여부
4,Q020,금속 칸막이·벽·천장·지붕 반대면 가연물의 열전도·복사 착화 우려 여부
5,Q025,초기소화설비 비치·접근 가능 여부
6,Q026,작업 종료 후 잔불·불티 확인 여부


In [9]:
def score_numeric_answer(question_id, value):
    """단위 변환이 필요 없는 숫자형 질문을 점수로 변환합니다."""

    question_id = str(question_id).strip()

    # 1. 질문 확인
    matched_questions = questions.loc[
        questions["항목ID"].str.strip() == question_id
    ]

    if len(matched_questions) != 1:
        raise ValueError(f"질문ID가 없거나 중복됩니다: {question_id}")

    question = matched_questions.iloc[0]

    if question["입력형식"].strip() != "숫자":
        raise ValueError("숫자형 질문만 처리할 수 있습니다.")

    # 가스농도처럼 단위 선택·환산이 필요한 입력은 별도 처리
    unit = question["단위"].strip()

    if "/" in unit:
        raise NotImplementedError(
            "여러 단위를 사용하는 질문은 단위 변환 기능이 필요합니다."
        )

    # 2. 입력값 검증
    if isinstance(value, bool):
        raise ValueError("True/False 대신 숫자를 입력하세요.")

    try:
        number = float(value)
    except (TypeError, ValueError):
        raise ValueError("숫자를 입력하세요.") from None

    if not math.isfinite(number):
        raise ValueError("유한한 숫자를 입력하세요.")

    # 현재 CSV의 숫자 입력 제한 해석
    allowed = question["선택지·허용값"].strip()

    if allowed == "0 이상":
        if number < 0:
            raise ValueError("0 이상의 숫자를 입력하세요.")
    else:
        raise NotImplementedError(
            f"아직 처리하지 않는 입력 제한입니다: {allowed}"
        )

    # 3. 연결된 규칙 조회
    rule_id = question["점수규칙ID"].strip()

    candidates = score_rules.loc[
        score_rules["점수규칙ID"].str.strip() == rule_id
    ]

    matches = []

    for _, rule in candidates.iterrows():
        if rule["규칙유형"].strip() != "수치형":
            raise NotImplementedError(
                "파생변수·범위형 규칙은 이후 단계에서 처리합니다."
            )

        operator = rule["비교연산자"].strip()
        threshold = float(rule["기준값1"])

        if not math.isfinite(threshold):
            raise ValueError(f"{rule_id}의 기준값이 잘못되었습니다.")

        if operator == "<":
            matched = number < threshold
        elif operator == "<=":
            matched = number <= threshold
        elif operator == ">":
            matched = number > threshold
        elif operator == ">=":
            matched = number >= threshold
        elif operator == "EQ":
            matched = number == threshold
        else:
            raise NotImplementedError(
                f"지원하지 않는 비교연산자입니다: {operator}"
            )

        if matched:
            matches.append(rule)

    if len(matches) != 1:
        raise ValueError(
            f"일치하는 규칙이 없거나 중복됩니다: {rule_id}, {number}"
        )

    # 4. CSV의 점수와 가중치로 계산
    rule = matches[0]

    score = float(rule["점수"])
    weight = float(rule["가중치"])
    max_score = float(rule["정규화최대점수"])

    if not all(math.isfinite(x) for x in [score, weight, max_score]):
        raise ValueError("점수·가중치·정규화 기준을 확인하세요.")

    if max_score <= 0 or weight < 0 or not 0 <= score <= max_score:
        raise ValueError("점수 또는 가중치가 허용 범위를 벗어났습니다.")

    included = weight > 0

    return {
        "항목ID": question_id,
        "확인항목": question["확인항목"],
        "사용자응답": number,
        "단위": unit,
        "점수규칙ID": rule_id,
        "조건순번": rule["조건순번"],
        "점수": score,
        "가중치": weight,
        "가중점수": score * weight if included else 0,
        "최대가중점수": max_score * weight if included else 0,
        "점수합산대상": included,
        "적용조건": rule["적용조건"],
        "확정상태": rule["확정상태"],
    }

In [11]:
distance_results = [
    score_numeric_answer("Q002", distance)
    for distance in [0, 10.9, 11, 15]
]

display(pd.DataFrame(distance_results))

,항목ID,확인항목,사용자응답,단위,점수규칙ID,조건순번,점수,가중치,가중점수,최대가중점수,점수합산대상,적용조건,확정상태
0,Q002,가연물과 작업점 거리,0.0,m,S002,1,1.0,1.5,1.5,4.5,True,화기·불티 작업,미확정
1,Q002,가연물과 작업점 거리,10.9,m,S002,1,1.0,1.5,1.5,4.5,True,화기·불티 작업,미확정
2,Q002,가연물과 작업점 거리,11.0,m,S002,2,0.0,1.5,0.0,4.5,True,화기·불티 작업,미확정
3,Q002,가연물과 작업점 거리,15.0,m,S002,2,0.0,1.5,0.0,4.5,True,화기·불티 작업,미확정


In [12]:
def calculate_task_risk(task_id, answers):
    """
    선택형·숫자형 응답을 함께 계산합니다.
    적용조건과 법령 경고는 아직 검사하지 않는 시험 계산입니다.
    """

    # 1. 필수응답 누락과 작업 외 질문 확인
    check_result, missing = check_required_answers(task_id, answers)

    if check_result["작업외질문ID"]:
        raise ValueError(
            f"해당 작업의 질문이 아닙니다: "
            f"{check_result['작업외질문ID']}"
        )

    if not missing.empty:
        raise ValueError(
            f"필수응답이 누락됐습니다: "
            f"{missing['항목ID'].tolist()}"
        )

    # 2. 질문별 입력형식에 맞는 함수 호출
    results = []

    for question_id, answer in answers.items():
        question = questions.loc[
            questions["항목ID"].str.strip() == question_id
        ]

        if len(question) != 1:
            raise ValueError(f"질문ID가 없거나 중복됩니다: {question_id}")

        question = question.iloc[0]
        input_type = question["입력형식"].strip()

        if input_type == "선택":
            result = score_answer(question_id, answer)
        elif input_type == "숫자":
            result = score_numeric_answer(question_id, answer)
        else:
            raise NotImplementedError(
                f"지원하지 않는 입력형식입니다: {input_type}"
            )

        group = question["중복집계그룹"].strip()
        method = question["그룹집계방식"].strip()

        if not group or method != "MAX_PER_TASK":
            raise ValueError(
                f"{question_id}의 중복집계 설정을 확인하세요."
            )

        result["중복집계그룹"] = group
        results.append(result)

    detail = pd.DataFrame(results)

    # 3. 합산 대상 중 그룹별 대표 응답 선택
    # 가중점수가 높은 응답을 사용하고, 동점이면 항목ID 순으로 선택
    candidates = detail.loc[detail["점수합산대상"]].copy()

    representatives = (
        candidates
        .sort_values(
            ["가중점수", "항목ID"],
            ascending=[False, True],
        )
        .drop_duplicates(subset="중복집계그룹", keep="first")
    )

    detail["집계대표여부"] = detail.index.isin(
        representatives.index
    )

    # 대표 응답의 분자와 분모를 함께 합산
    total_score = representatives["가중점수"].sum()
    max_score = representatives["최대가중점수"].sum()

    if max_score > 0:
        risk_index = total_score / max_score * 100
        status = "시험 계산 완료 — 적용조건·법령 미검증"
    else:
        risk_index = None
        status = "계산 불가: 합산 대상 없음"

    summary = {
        "작업ID": task_id,
        "입력질문수": len(detail),
        "집계항목수": len(representatives),
        "총가중위험점수": total_score,
        "최대가중점수": max_score,
        "위험지수": risk_index,
        "계산상태": status,
    }

    return summary, detail

In [13]:
h05_answers = {
    "Q001": "있음",
    "Q002": 5,
    "Q003": "미실시",
    "Q004": "미배치",
    "Q018": "용접방화포 미사용",
    "Q019": "없음",
    "Q020": "없음",
    "Q025": "적정",
    "Q026": "완료",
}

summary, detail = calculate_task_risk("H05", h05_answers)

display(detail)
display(pd.DataFrame([summary]))

,항목ID,확인항목,사용자응답,점수규칙ID,조건순번,점수,가중치,가중점수,최대가중점수,점수합산대상,적용조건,확정상태,중복집계그룹,단위,집계대표여부
0,Q001,주변 가연물 존재 여부,있음,S001,1,1.0,1.5,1.5,4.5,True,화기·불티/전기/열원 작업 공통,미확정,Q001,NaN,True
1,Q002,가연물과 작업점 거리,5.0,S002,1,1.0,1.5,1.5,4.5,True,화기·불티 작업,미확정,Q002,m,True
2,Q003,불티 비산방지조치(용접방화포 등) 실시 여부,미실시,S003,1,3.0,1.5,4.5,4.5,True,화기·불티 작업,미확정,Q003,NaN,True
3,Q004,화재감시자 배치 여부,미배치,S004,1,3.0,2.0,6.0,6.0,True,법령상 배치대상 조건 충족 시 적용,미확정,Q004,NaN,True
4,Q018,용접방화포 사용 시 성능인증 제품 여부,용접방화포 미사용,S019,3,0.0,0.0,0.0,0.0,False,다른 적정 비산방지조치를 적용하는 경우,미확정,Q018,NaN,False
5,Q019,작업점 하부 가연물의 불티 착화 우려 여부,없음,S020,2,0.0,1.0,0.0,3.0,True,다른 배치조건도 함께 확인,미확정,Q019,NaN,True
6,Q020,금속 칸막이·벽·천장·지붕 반대면 가연물의 열전도·복사 착화 우려 여부,없음,S021,2,0.0,1.0,0.0,3.0,True,다른 배치조건도 함께 확인,미확정,Q020,NaN,True
7,Q025,초기소화설비 비치·접근 가능 여부,적정,S026,1,0.0,1.5,0.0,4.5,True,화재위험 작업,미확정,Q025,NaN,True
8,Q026,작업 종료 후 잔불·불티 확인 여부,완료,S027,1,0.0,1.0,0.0,3.0,True,용접·용단·그라인딩 등 화기·불티 작업,미확정,Q026,NaN,True


,작업ID,입력질문수,집계항목수,총가중위험점수,최대가중점수,위험지수,계산상태
0,H05,9,8,13.5,33.0,40.909091,시험 계산 완료 — 적용조건·법령 미검증


In [14]:
def get_related_standards(detail, count=3):
    """주요 위험요인별로 연결된 기준과 필수조치를 조회합니다."""

    top_risks = get_top_risks(detail, count=count)
    results = []

    for _, risk in top_risks.iterrows():
        # 예: "R001;R014"를 ["R001", "R014"]로 분리
        standard_ids = list(dict.fromkeys(
            split_ids(risk["관련기준ID"])
        ))

        for standard_id in standard_ids:
            # 연결된 기준이 정확히 한 건인지 확인
            standard = get_single_row(
                standards.loc[
                    standards["기준ID"] == standard_id
                ],
                f"기준 {standard_id}",
            )

            results.append({
                "항목ID": risk["항목ID"],
                "위험요인": risk["확인항목"],
                "사용자응답": risk["사용자응답"],
                "가중점수": risk["가중점수"],
                "기준ID": standard_id,
                "기준구분": standard["구분"],
                "적용조건": standard["적용조건"],
                "필수조치": standard["필수조치"],
                "법령·가이드명": standard["법령_가이드명"],
                "조문·문서번호": standard["조문_문서번호"],
                "출처URL": standard["출처URL"],
                "판정상태": "관련 근거 조회 — 적용·위반 여부 미검증",
            })

    columns = [
        "항목ID", "위험요인", "사용자응답", "가중점수",
        "기준ID", "기준구분", "적용조건", "필수조치",
        "법령·가이드명", "조문·문서번호", "출처URL", "판정상태",
    ]

    return pd.DataFrame(results, columns=columns)

In [15]:
# 기존 H05 가상 응답으로 계산
summary, detail = calculate_task_risk("H05", h05_answers)

# 주요 위험요인 3개에 연결된 근거 조회
related_standards = get_related_standards(detail, count=3)

display(pd.DataFrame([summary]))

# 긴 설명을 읽기 쉽게 표시
with pd.option_context("display.max_colwidth", 120):
    display(related_standards)

NameError: name 'get_top_risks' is not defined